[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/energy-system-modeling/blob/main/notebooks/p1_foundations/02_one_house_balance.ipynb)

# An energy balance for one house
## REE 4301 / IE 5300 - Energy Systems Modeling

A small, complete version of what SB1 asks for, so you can see the shape before you build your own. It is deliberately short - your system will have different flows, but the same five moves.

The diagram at the end is the same kind of picture as the [LLNL energy flow charts](https://flowcharts.llnl.gov): sources on the left, what they were used for in the middle, and useful against rejected energy on the right.


## Setup

One cell, the same in every notebook here: it installs what Colab does not have, fetches this repository so `data/` and `src/` are present, and moves into this notebook's own folder so the relative paths below resolve. The notebook's own imports follow in the same cell.


In [1]:
# --- setup: generated by tools/sync_setup_cells.py -- do not edit here, edit that
# The same cell in every notebook in this series. It installs what Colab does
# not have, fetches the repository so that data/ and src/ are present, and moves
# into this notebook's own folder so the relative paths below resolve.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sear-labs/energy-system-modeling"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1]
NOTEBOOK_DIR = "notebooks/p1_foundations"

# Pinned, per Part 1 rule 3: an unpinned install will one day pull a major
# version with a changed API and either break or silently alter the answer.
PINS = []

if "google.colab" in sys.modules:
    if PINS:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    # An ABSOLUTE base, so re-running this cell is safe. Colab's "Run all" is
    # commonly run twice, and a relative check would look for the clone inside
    # the folder it had already moved into -- cloning a second copy nested one
    # level down, then working from the wrong one.
    BASE = Path("/content") if Path("/content").is_dir() else Path.home()
    REPO_DIR = BASE / REPO_NAME
    if not REPO_DIR.exists():
        cloned = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            capture_output=True, text=True)
        if cloned.returncode != 0:
            raise SystemExit(
                "Could not clone " + REPO_URL + "\n"
                + (cloned.stderr or "").strip() + "\n\n"
                "If that says 'not found', the repository is still private.\n"
                "A raw file URL fails the same way, so there is no way around\n"
                "it: it has to be public before a student can run this.")
    os.chdir(REPO_DIR / NOTEBOOK_DIR)
elif Path.cwd().name != Path(NOTEBOOK_DIR).name:
    raise SystemExit(
        "Run this notebook from its own folder (" + NOTEBOOK_DIR + "),\n"
        "so that ../../src and ../../data resolve.")

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
print("working directory:", Path.cwd().name)

# --- end generated setup; the notebook's own imports follow ---


import pandas as pd
import plotly.graph_objects as go

working directory: p1_foundations


---
## 1. The boundary, in one sentence

*Everything that crosses the property line of one house over one year: electricity through the meter, gas through the pipe, and gasoline bought for the car parked there.*

Write yours the same way, and be as specific. Most of the trouble in this assignment comes from a boundary that was never actually stated.


## 2. What came in, in the units it was billed in

Bills do not arrive in one unit. Electricity is kWh, gas is therms or ccf, gasoline is gallons - so the conversion factors go in the table, where they can be checked.


In [2]:
KWH_PER_THERM = 29.3      # 1 therm = 100,000 Btu
KWH_PER_GALLON = 33.7     # gasoline, lower heating value

inputs = pd.DataFrame([
    ['grid electricity', 10_000, 'kWh',    1.0,            'utility bill'],
    ['natural gas',         600, 'therms', KWH_PER_THERM,  'gas bill'],
    ['gasoline',            500, 'gallons', KWH_PER_GALLON, 'fuel receipts'],
], columns=['source', 'as billed', 'unit', 'kWh per unit', 'where from'])

inputs['kWh'] = inputs['as billed'] * inputs['kWh per unit']
print(inputs.to_string(index=False))
print(f"\ntotal energy in: {inputs['kWh'].sum():,.0f} kWh")


          source  as billed    unit  kWh per unit    where from     kWh
grid electricity      10000     kWh           1.0  utility bill 10000.0
     natural gas        600  therms          29.3      gas bill 17580.0
        gasoline        500 gallons          33.7 fuel receipts 16850.0

total energy in: 44,430 kWh


## 3. What it was used for, and how much of it did the job

Every conversion loses something. The efficiencies below are rough figures for a furnace, a car and a mixed electrical load - find better ones for your own system and say where you got them.


In [3]:
efficiency = {'grid electricity': 0.90,   # lights, appliances, some heat
              'natural gas': 0.85,        # a decent furnace
              'gasoline': 0.25}           # tank to wheels

flows = inputs[['source', 'kWh']].copy()
flows['efficiency'] = flows['source'].map(efficiency)
flows['useful'] = flows['kWh'] * flows['efficiency']
flows['rejected'] = flows['kWh'] - flows['useful']

cols = {'kWh': 0, 'efficiency': 2, 'useful': 0, 'rejected': 0}
print(flows.round(cols).to_string(index=False))


          source     kWh  efficiency  useful  rejected
grid electricity 10000.0        0.90  9000.0    1000.0
     natural gas 17580.0        0.85 14943.0    2637.0
        gasoline 16850.0        0.25  4212.0   12638.0


## 4. Does it close?

In minus out. Here it closes exactly, because the efficiencies were *assumed* rather than measured - so the rejected column was calculated as the leftover.

Your balance will not close exactly, and that is the interesting part. Report the residual and say what it is.


In [4]:
total_in = flows['kWh'].sum()
useful = flows['useful'].sum()
rejected = flows['rejected'].sum()

print(f'in        {total_in:>9,.0f} kWh')
print(f'useful    {useful:>9,.0f} kWh   {useful / total_in:.0%}')
print(f'rejected  {rejected:>9,.0f} kWh   {rejected / total_in:.0%}')
print(f'residual  {total_in - useful - rejected:>9,.0f} kWh')


in           44,430 kWh
useful       28,156 kWh   63%
rejected     16,274 kWh   37%
residual          0 kWh


---
## 5. Draw it

Widths come from the numbers above. Nothing is drawn by hand.


In [5]:
labels = list(flows['source']) + ['useful energy', 'rejected energy']
i_useful, i_rejected = len(flows), len(flows) + 1

source_idx, target_idx, value = [], [], []
for i, row in flows.iterrows():
    source_idx += [i, i]
    target_idx += [i_useful, i_rejected]
    value += [row['useful'], row['rejected']]

fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=20, thickness=20,
              color=['#4C72B0', '#DD8452', '#937860', '#55A868', '#C44E52']),
    link=dict(source=source_idx, target=target_idx, value=value),
))
fig.update_layout(title_text='One house, one year (kWh)',
                  font_size=12, height=420)

# Reused verbatim from the CARES Book figure manifest (F9_house_sankey),
# where this same diagram is already described and reviewed. Kept identical
# on purpose: two descriptions of one figure drift, and a reader meeting it
# in the book and in the notebook should be told the same thing.
ALT_TEXT = (
    'A Sankey diagram of one house over one year. Grid electricity, '
    'natural gas and gasoline enter on the left and split into useful '
    'and rejected energy on the right, with gasoline contributing by far '
    'the largest share of the rejected block.')

fig.show()


### Read your own diagram

Look at the rejected block and work out which source is feeding most of it. Then check:


In [6]:
share = (flows.set_index('source')['rejected'] / rejected).sort_values(
    ascending=False)
print('share of all rejected energy\n')
print(share.map('{:.0%}'.format).to_string())


share of all rejected energy

source
gasoline            78%
natural gas         16%
grid electricity     6%


> One source produces most of the waste in this house, and it is not the one with the largest bill. Why?

> This house comes out around 63% efficient. The LLNL chart puts the whole United States at roughly a third. Same physics, very different number - what is inside their boundary that is outside yours?


---
### Does the package agree?

`esm.balance` computes this same balance from `data/raw/house_energy.csv`. **It is not a second solver, because there is nothing here to solve.** Re-doing `billed x factor x efficiency` a second way would just be a second copy of the same three multiplications, and a wrong conversion factor would sit in both copies quite happily.

So the checks that carry weight here are a different kind, and the difference is worth more than the numbers:

1. **Conservation, exactly.** Useful plus rejected must equal input, per source and in total. Arithmetic cannot get this right by accident.

2. **The units, rebuilt from primary definitions.** The real risk in a notebook about units is a conversion factor, so that is what the second implementation attacks. A therm *is* 100,000 Btu, and the Btu is exactly defined, so that factor has a true value and can be checked.

**And one factor deliberately is not checked.** There is no exact kWh per gallon of gasoline: heating value is measured, and varies by blend and standard. The package says so out loud rather than leaving a silent gap, because asserting a tolerance against a true value that does not exist would be inventing precision. Notice which of your own numbers are definitions and which are measurements - it is the same question as asking which of them you are allowed to check.


In [7]:
from esm.balance import (BTU_PER_THERM, conservation_residual,
                        load_balance_instance, si_consistency,
                        solve_balance, unverifiable_factors)
from esm.tolerance import AGREEMENT_RTOL, relative

inst = load_balance_instance()
packaged = solve_balance(inst)

checks = [('energy in', total_in, packaged.total_in),
          ('useful', useful, packaged.total_useful),
          ('rejected', rejected, packaged.total_rejected)]
checks += [(f'{s} in', float(flows.set_index('source')['kWh'][s]),
            packaged.kwh_in[s]) for s in packaged.kwh_in]

print(f'{"quantity":26s} {"notebook":>12s} {"package":>12s} {"rel diff":>10s}')
for label, hand, pkg in checks:
    print(f'{label:26s} {hand:12,.1f} {pkg:12,.1f} {relative(hand, pkg):10.1e}')

worst = max(relative(a, b) for _, a, b in checks)
assert worst < AGREEMENT_RTOL, (
    f'notebook and package disagree by {worst:.2e}, '
    f'which is worse than {AGREEMENT_RTOL:.0e}')
print()
print(f'notebook and package agree to {worst:.1e}')

# conservation, independently recomputed
print()
print('conservation residual, per source:')
for name, res in conservation_residual(packaged).items():
    print(f'  {name:24s} {res:+.2e} kWh')

# the units, rebuilt from what a therm IS
print()
print(f'conversion factors with an exact definition'
      f' (1 therm = {BTU_PER_THERM:,} Btu):')
for name, (got, want, gap) in si_consistency(inst).items():
    print(f'  {name:24s} table {got:<8} exact {want:<14.8f} gap {gap:.1e}')
for name, unit in unverifiable_factors(inst).items():
    print(f'  {name:24s} {unit} has no exact value - not checked')


quantity                       notebook      package   rel diff
energy in                      44,430.0     44,430.0    0.0e+00
useful                         28,155.5     28,155.5    0.0e+00
rejected                       16,274.5     16,274.5    0.0e+00
grid electricity in            10,000.0     10,000.0    0.0e+00
natural gas in                 17,580.0     17,580.0    0.0e+00
gasoline in                    16,850.0     16,850.0    0.0e+00

notebook and package agree to 0.0e+00

conservation residual, per source:
  grid electricity         +0.00e+00 kWh
  natural gas              +0.00e+00 kWh
  gasoline                 +0.00e+00 kWh
  total                    +0.00e+00 kWh

conversion factors with an exact definition (1 therm = 100,000 Btu):
  grid electricity         table 1.0      exact 1.00000000     gap 0.0e+00
  natural gas              table 29.3     exact 29.30710702    gap 2.4e-04
  gasoline                 gallons has no exact value - not checked


---
## What you do for your own system

Same five moves, your own numbers:

1. State the boundary in one sentence.
2. Table every flow in the unit it was measured in, with the conversion factor and the source alongside.
3. Split each flow into what did the job and what did not.
4. Report the residual, and say what it is rather than adjusting it away.
5. Generate the diagram from the table.

A car, a single power plant, a building and a small factory all work. Pick something you can find real numbers for.


*Before class: this balance was drawn around one house. If you drew it around the power station instead, which flows would move from outside the boundary to inside it?*
